In [1]:
import transformers
import hydra
import pyrootutils
import logging
import torch
import torch.distributed as dist
import wandb
import io
import numpy as np
import os
from petrel_client.client import Client
from src.model import *
from src.train.trainer import CustomTrainer
from typing import Optional, List
from dataclasses import dataclass, field
from omegaconf import OmegaConf
from src.share_utils.py_config_utils import Config, eval_dict_leaf
from src.train.cap_utils import compute_metrics_cider
from src.train.qa_utils import compute_metrics_vqa
from src.share_utils.distributed import is_main_process
from src.share_utils.logger import setup_logger
from src.train.utils import init_distributed_mode, setup_output_dir
from src.share_utils.serialize import local_broadcast_process_authkey
import deepspeed
from src.tokenizer import MultimodalLlamaTokenizer
from src.share_utils.special_token import special_tokens
from src.model.base_model import LLMConfig
from src.dataset.it_dataset_mistral_eval import ITTrainDataset_mistral_eval
from torch.utils.data import DataLoader
from src.share_utils.my_easydict import MyEasyDict as edict
from src.dataset.create import get_train_transform
from src.train.trainer import collate_hd

/mnt/petrelfs/yanziang/xformers/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/mnt/petrelfs/yanziang/xformers/xformers/ops/fmha/flash.py:338: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/mnt/petrelfs/yanziang/xformers/xformers/ops/swiglu_op.py:128: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(cls, ctx, x, w1, b1, w2, b2, w3, b3):
/mnt/petrelfs/yanziang/xformers/xformers/ops/swiglu_op.py:149: FutureWarning: `torch.cuda.amp.custom_bwd(args...

[2024-11-18 14:39:26,375] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/transformers/deepspeed.py:23: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(
/mnt/petrelfs/yanziang/.conda/envs/mllm/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-devel package with yum
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.4
 [WARNING]  using untested triton version (3.0.0), only 1.0.0 is known to be compatible


/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):


In [8]:
tokenizer = MultimodalLlamaTokenizer.from_pretrained(
    "/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2", 
    local_files_only=True,
    n_query=96,
    v_query=96,
    special_tokens = special_tokens, 
) 
# print(cfg_path.model)
model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/expert_umtl_f8_s2.py'))


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'MultimodalLlamaTokenizer'.


In [4]:
tokenizer = MultimodalLlamaTokenizer.from_pretrained(
    "/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2", 
    local_files_only=True,
    n_query=96,
    v_query=96,
    special_tokens = special_tokens, 
) 
# print(cfg_path.model)
model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/expert_umtl_f8_s2.py'))

from src.model.base_model import LLMConfig
config = LLMConfig.from_pretrained('/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2', trust_remote_code=True)
config.model_config = model_cfg.model
config.model_tokenizer = tokenizer
from src.dataset.videoload import (
    GroupNormalize, GroupScale, GroupCenterCrop, 
    Stack, ToTorchFormatTensor
)
from src.tokenizer.multimodal_llama_tokenizer import MultimodalLlamaTokenizer
from src.dataset.it_dataset_mistral_new import ITTrainDataset_mistral_new
from src.train.trainer import collate_hd
# tokenizer = MultimodalLlamaTokenizer.from_pretrained("/mnt/petrelfs/wangchenting/Mistral-7B-Instruct-v0.2", local_files_only=True) 
from src.share_utils.special_token import special_tokens
tokenizer = MultimodalLlamaTokenizer.from_pretrained(
    "/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2", 
    local_files_only=True,
    n_query=96,
    v_query=96,
    special_tokens = special_tokens, 
) 
from src.share_utils.my_easydict import MyEasyDict as edict
from torchvision import transforms
from src.dataset.create import get_train_transform, create_dataset, create_loader, create_sampler
from src.model.multimodalllm_pt import MultiModalLLM_PT
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
from src.share_utils.py_config_utils import Config, eval_dict_leaf

anno_root_it = "/mnt/petrelfs/share_data/videointern/annotations/anno_instruction/videochat_new"
import torch
# torch.backends.cudnn.enabled = False

import argparse
def initialize_parser():
    
    parser = argparse.ArgumentParser()
    
    # Arguments
    parser.add_argument("--dset_name", type=str, default='charadesSTA', choices=["hl", 'charadesSTA'])
    parser.add_argument("--eval_split_name", type=str, default="val",
                        help="should match keys in video_duration_idx_path, must set for VCMR")
    parser.add_argument("--data_ratio", type=float, default=1.0,
                        help="how many training and eval data to use. 1.0: use all, 0.1: use 10%.")
    parser.add_argument("--results_root", type=str, default="results")
    parser.add_argument("--exp_id", type=str, default=None, help="id of this run, required at training")
    parser.add_argument("--max_es_cnt", type=int, default=200, help="number of epochs to early stop")
    parser.add_argument("--eval_epoch", type=int, default=5, help="inference epoch")
    parser.add_argument("--grad_clip", type=float, default=0.1, help="perform gradient clip, -1: disable")
    parser.add_argument("--eval_untrained", action="store_true", help="Evaluate on un-trained model")
    parser.add_argument("--resume_all", action="store_true", help="if --resume_all, load optimizer/scheduler/epoch as well")
    parser.add_argument("--start_epoch", type=int, default=None, help="if None, will be set automatically when using --resume_all")

    # Data config
    parser.add_argument("--max_q_l", type=int, default=-1)
    parser.add_argument("--max_v_l", type=int, default=-1)
    parser.add_argument("--clip_length", type=float, default=1.0)
    parser.add_argument("--max_windows", type=int, default=5)
    parser.add_argument("--train_path", type=str, default=None)
    parser.add_argument("--eval_path", type=str, default=None, help="Evaluating during training")
    parser.add_argument("--no_norm_vfeat", action="store_true", help="Do not normalize video feat")
    parser.add_argument("--no_norm_tfeat", action="store_true", help="Do not normalize text feat")
    parser.add_argument("--v_feat_dirs", type=str, nargs="+", help="video feature dirs")
    parser.add_argument("--t_feat_dir", type=str, help="text/query feature dir")
    parser.add_argument("--v_feat_dim", type=int, default=770, help="video feature dim")
    parser.add_argument("--t_feat_dim", type=int, default=4096, help="text/query feature dim")
    parser.add_argument("--ctx_mode", type=str, default="video_tef")

    # Model config
    parser.add_argument('--position_embedding', default='sine', type=str, choices=('sine', 'learned'),
                        help="Type of positional embedding to use")
    parser.add_argument('--enc_layers', default=3, type=int, help="Number of encoding layers in the transformer")
    parser.add_argument('--dec_layers', default=3, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--t2v_layers', default=2, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--sent_layers', default=1, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--moment_layers', default=1, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--dummy_layers', default=2, type=int, help="Number of encoding layers in the transformer")
    parser.add_argument('--dim_feedforward', default=1024, type=int, help="Intermediate size of the feedforward layers")
    parser.add_argument('--hidden_dim', default=256, type=int, help="Size of the embeddings")
    parser.add_argument('--input_dropout', default=0.5, type=float, help="Dropout applied in input")
    parser.add_argument('--dropout', default=0.1, type=float, help="Dropout applied in the transformer")
    parser.add_argument("--txt_drop_ratio", default=0, type=float, help="drop tokens from text input")
    parser.add_argument("--use_txt_pos", action="store_true", help="use position_embedding for text as well.")
    parser.add_argument('--nheads', default=8, type=int, help="Number of attention heads inside the transformer")
    parser.add_argument('--num_queries', default=10, type=int, help="Number of query slots")
    parser.add_argument('--num_dummies', default=45, type=int, help="Number of dummy tokens")
    parser.add_argument('--total_prompts', default=10, type=int, help="Number of query slots")
    parser.add_argument('--num_prompts', default=1, type=int, help="Number of dummy tokens")
    parser.add_argument('--pre_norm', action='store_true')
    parser.add_argument("--n_input_proj", type=int, default=2, help="#layers to encoder input")
    parser.add_argument("--contrastive_hdim", type=int, default=64, help="dim for contrastive embeddings")
    parser.add_argument("--temperature", type=float, default=0.07, help="temperature for contrastive loss")

    # Loss
    parser.add_argument("--saliency_margin", type=float, default=0.2)
    parser.add_argument('--no_aux_loss', dest='aux_loss', action='store_false', help="Disables auxiliary decoding losses")
    parser.add_argument("--span_loss_type", default="l1", type=str, choices=['l1', 'ce'],
                        help="Loss type for span")
    parser.add_argument("--contrastive_align_loss", action="store_true",
                        help="Disable contrastive alignment loss")
    parser.add_argument('--set_cost_span', default=10, type=float, help="L1 span coefficient in the matching cost")
    parser.add_argument('--set_cost_giou', default=1, type=float, help="GIoU span coefficient in the matching cost")
    parser.add_argument('--set_cost_class', default=4, type=float, help="Class coefficient in the matching cost")

    # Loss coefficients
    parser.add_argument("--lw_saliency", type=float, default=4, help="weight for saliency loss")
    parser.add_argument("--lw_wattn", type=float, default=1., help="weight for attention loss")
    parser.add_argument("--lw_ms_align", type=float, default=1., help="weight for multi-scale alignment loss")
    parser.add_argument("--lw_distill", type=float, default=1., help="weight for distillation loss")
    parser.add_argument('--span_loss_coef', default=10, type=float)
    parser.add_argument('--giou_loss_coef', default=1, type=float)
    parser.add_argument('--label_loss_coef', default=4, type=float)
    parser.add_argument('--eos_coef', default=0.1, type=float,
                        help="Relative classification weight of the no-object class")
    parser.add_argument("--contrastive_align_loss_coef", default=0.02, type=float)

    parser.add_argument("--no_sort_results", action="store_true", help="do not sort results")
    parser.add_argument("--max_before_nms", type=int, default=10)
    parser.add_argument("--max_after_nms", type=int, default=10)
    parser.add_argument("--conf_thd", type=float, default=0.0, help="only keep windows with conf >= conf_thd")
    parser.add_argument("--nms_thd", type=float, default=-1,
                        help="additional use of non-maximum suppression")

    return parser


from src.model.base_model import LLMConfig
config = LLMConfig.from_pretrained('/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2', trust_remote_code=True)

cg_opt = initialize_parser()
model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/expert_umtl_f8_s2.py'))
config.model_config = model_cfg.model
config.cg_opt = cg_opt.parse_args([])
config.model_tokenizer = tokenizer

from src.share_utils.distributed import get_world_size, get_rank, gather_together
from src.dataset.dataloader import MetaLoader
from src.dataset.create import get_media_types
from src.dataset.create import create_dataset

path = '/mnt/petrelfs/share_data/yanziang/reasoning_chat2'
model = MultiModalLLM_PT.from_pretrained(path, config=config, local_files_only=True)
device = f'cuda:{0}'
model = model.to(dtype=torch.bfloat16)
model = model.cuda()

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'MultimodalLlamaTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'MultimodalLlamaTokenizer'.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

trainable params: 42,520,688 || all params: 7,284,310,128 || trainable%: 0.5837


/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.va

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of MultiModalLLM_PT were not initialized from the model checkpoint at /mnt/petrelfs/share_data/yanziang/reasoning_chat2 and are newly initialized: ['sam.memory_encoder.fuser.layers.0.gamma', 'sam.memory_encoder.fuser.layers.1.gamma']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:

def get_prompt(conv):
    ret = conv.system + conv.sep
    for role, message in conv.messages:
        if message:
            ret += role + " " + message + " " + conv.sep
        else:
            ret += role
    return ret


def get_prompt2(conv):
    ret = conv.system + conv.sep
    count = 0
    for role, message in conv.messages:
        count += 1
        if count == len(conv.messages):
            ret += role + " " + message
        else:
            if message:
                ret += role + " " + message + " " + conv.sep
            else:
                ret += role
    return ret


def get_context_emb(conv, model, img_list, answer_prompt=None, print_res=False):
    if answer_prompt:
        prompt = get_prompt2(conv)
    else:
        prompt = get_prompt(conv)
    if print_res:
        print(prompt)
    if '<VideoHere>' in prompt:
        prompt_segs = prompt.split('<VideoHere>')
    else:
        prompt_segs = prompt.split('<ImageHere>')
    assert len(prompt_segs) == len(img_list) + 1, "Unmatched numbers of image placeholders and images."
    with torch.no_grad():
        seg_tokens = [
            model.tokenizer(
                seg, return_tensors="pt", add_special_tokens=i == 0).to("cuda:0").input_ids
            # only add bos to the first seg
            for i, seg in enumerate(prompt_segs)
        ]
        seg_embs = [model.lm.base_model.model.model.embed_tokens(seg_t) for seg_t in seg_tokens]
    mixed_embs = [emb for pair in zip(seg_embs[:-1], img_list) for emb in pair] + [seg_embs[-1]]
    mixed_embs = torch.cat(mixed_embs, dim=1)
    return mixed_embs


def ask(text, conv):
    conv.messages.append([conv.roles[0], text])

from transformers import StoppingCriteria, StoppingCriteriaList

class StoppingCriteriaSub(StoppingCriteria):
    def __init__(self, stops=[], encounters=1):
        super().__init__()
        self.stops = stops
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        for stop in self.stops:
            if torch.all((stop == input_ids[0][-len(stop):])).item():
                return True
        return False


from transformers import StoppingCriteria, StoppingCriteriaList

    
def answer(conv, model, img_list, do_sample=True, max_new_tokens=200, num_beams=1, min_length=1, top_p=0.9,
               repetition_penalty=1.0, length_penalty=1, temperature=1.0, answer_prompt=None, print_res=False):
    stop_words_ids = [
        torch.tensor([2]).to("cuda:0"),
        torch.tensor([29871, 2]).to("cuda:0")]  # '</s>' can be encoded in two different ways.
    stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])
    
    conv.messages.append([conv.roles[1], answer_prompt])
    embs = get_context_emb(conv, model, img_list, answer_prompt=answer_prompt, print_res=print_res)
    with torch.no_grad():
        outputs = model.lm.generate(
            inputs_embeds=embs,
            max_new_tokens=max_new_tokens,
            stopping_criteria=stopping_criteria,
            num_beams=num_beams,
            do_sample=do_sample,
            min_length=min_length,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            length_penalty=length_penalty,
            temperature=temperature,
        )
    output_token = outputs[0]
    if output_token[0] == 0:  # the model might output a unknow token <unk> at the beginning. remove it
            output_token = output_token[1:]
    if output_token[0] == 1:  # some users find that there is a start token <s> at the beginning. remove it
            output_token = output_token[1:]
    output_text = model.tokenizer.decode(output_token, add_special_tokens=False)
    output_text = output_text.split('</s>')[0]  # remove the stop sign </s>
#     output_text = output_text.split('[/INST]')[-1].strip()
    conv.messages[-1][1] = output_text + '</s>'
    return output_text, output_token.cpu().numpy()
import numpy as np
def get_index(num_frames, num_segments):
    seg_size = float(num_frames - 1) / num_segments
    start = int(seg_size / 2)
    offsets = np.array([
        start + int(np.round(seg_size * idx)) for idx in range(num_segments)
    ])
    return offsets

from torch.utils.data import Dataset
from torchvision.transforms.functional import InterpolationMode
from decord import VideoReader, cpu
import torchvision.transforms as T
from PIL import Image
def load_video(video_path, num_segments=8, return_msg=False, resolution=224):
    vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
    num_frames = len(vr)
    frame_indices = get_index(num_frames, num_segments)

    # transform
    crop_size = resolution
    scale_size = resolution
    input_mean = [0.48145466, 0.4578275, 0.40821073]
    input_std = [0.26862954, 0.26130258, 0.27577711]

    transform = T.Compose([
        GroupScale(int(scale_size), interpolation=InterpolationMode.BICUBIC),
        GroupCenterCrop(crop_size),
        Stack(),
        ToTorchFormatTensor(),
        GroupNormalize(input_mean, input_std) 
    ])

    images_group = list()
    for frame_index in frame_indices:
        img = Image.fromarray(vr[frame_index].numpy())
        images_group.append(img)
    torch_imgs = transform(images_group)
    if return_msg:
        fps = float(vr.get_avg_fps())
        sec = ", ".join([str(round(f / fps, 1)) for f in frame_indices])
        # " " should be added in the start and end
        msg = f"The video contains {len(frame_indices)} frames sampled at {sec} seconds."
        return torch_imgs, msg
    else:
        return torch_imgs


vid_path = "/mnt/petrelfs/yanziang/trackingllm/videoagent/demo_img/video_5008.mp4"
# vid_path = "./demo/example/jesse_dance.mp4"
vid_path = '/mnt/petrelfs/share_data/yanziang/videoagent-demo/object_s/7fca83bfd9c0e543291ebef5b6e481b5.mp4'
# from src.utils.easydict import EasyDict
from src.share_utils.my_easydict import MyEasyDict as EasyDict

num_frame = 16
resolution = 224
vid, msg = load_video(vid_path, num_segments=num_frame, return_msg=True, resolution=resolution)
# new_pos_emb = get_sinusoid_encoding_table(n_position=(resolution//16)**2*num_frame, cur_frame=num_frame)
# model.vision_encoder.encoder.pos_embed = new_pos_emb

print(msg)
    
# The model expects inputs of shape: T x C x H x W
TC, H, W = vid.shape
video = vid.reshape(1, TC//3, 3, H, W).to("cuda:0").to(torch.bfloat16)
            # B, T, C, H, W = image.shape
video = video.permute(0, 2, 1, 3, 4)
img_list = []
with torch.no_grad():
    image_emb = model.encode_vision(video, "Watch the video and answer the question.")
    image_emb = model.project_up(image_emb).to(torch.bfloat16)
    # image_emb, _ = model.encode_img(video, "Watch the video and answer the question.")
#     image_emb, _ = model.encode_img(video, "")

img_list.append(image_emb)
chat = EasyDict({
    "system": "",
    "roles": ("[INST]", "[/INST]"),
    "messages": [],
    "sep": ""
})

chat.messages.append([chat.roles[0], "<Video><VideoHere></Video> [/INST]"])
question = 'Describe the video.'
ask(question, chat)

llm_message = answer(conv=chat, model=model, do_sample=False, img_list=img_list, max_new_tokens=512, temperature=1e-5, print_res=True)[0]
llm_message

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The video contains 16 frames sampled at 0.7, 2.2, 3.7, 5.2, 6.7, 8.2, 9.7, 11.2, 12.7, 14.2, 15.8, 17.2, 18.8, 20.2, 21.8, 23.2 seconds.
[INST] <Video><VideoHere></Video> [/INST] [INST] Describe the video. [/INST]


' The video shows a person standing in front of a table with a pair of scissors and a cup. The person then proceeds to move the cup around the table while the scissors remain stationary. The video ends with the person standing in front of the table with the cup and scissors.'

In [ ]:
# demo for segmentation
import pycocotools.mask as mask_util
video_path = vid_path
exp = 'Cup with candy in the middle'

prompts = [{'exp':exp, "mask": "pnorm2:s3://mevis/masks_val_u/0.json", 'video_path':video_path}]

prompts.append({'exp':'cup with candy in middle', "mask": "pnorm2:s3://mevis/masks_val_u/0.json", 'video_path':video_path})
prompts.append({'exp':'cup which have a candy', "mask": "pnorm2:s3://mevis/masks_val_u/0.json", 'video_path':video_path})
prompts.append({'exp':'cup which have a candy inside', "mask": "pnorm2:s3://mevis/masks_val_u/0.json", 'video_path':video_path})

import json
with open('temp.json','w') as f:
    json.dump(prompts, f)


Myd = ITTrainDataset_mistral_eval(
    ann_file = edict(
        anno_path=f"temp.json", 
        data_root="",
        media_type='tracking2',
    ),
    transform=get_train_transform(config='', train_file='',mode='resize_noflip'),
    tokenizer=tokenizer,
    num_frames = 16
)


dataloader = DataLoader(
    Myd,
    batch_size=1,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
    collate_fn=collate_hd,
)

def generate(input_ids, attention_mask, labels, image, image_idx, video, video_idx, input_boxes, size_hw, sam_images):
    size_hw = [[720, 1280]]
    device = input_ids.device
    prompt = input_ids
    l_prompt = len(input_ids)
    temperature = 1e-5
    max_new_tokens = 20
    guide_w = 5
    stop_str = '</s>'
    bbox = []
    output_ids = list(input_ids[0])
    input_boxes = [box.cuda() for box in input_boxes]
    text_embeds = model.pad_text_embeds(input_ids=input_ids, image=image, video=video, image_idx=image_idx, video_idx=video_idx, return_visual=False, 
                                       instruction = None, output_boxes=None, input_boxes=input_boxes)
    for i in range(max_new_tokens):
        if i == 0:
            outputs = model.lm(
                    inputs_embeds=text_embeds,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    return_dict=True,
                )
            logits = outputs.logits
            past_key_values = outputs.past_key_values
        else:
            attention_mask = torch.ones(1, past_key_values[0][0].shape[-2] + 1, device=device)
            last_text_embeds = model.lm.get_input_embeddings()(torch.tensor(output_ids[-1], device=device).long()).detach().unsqueeze(0)
            last_text_embeds = last_text_embeds.unsqueeze(0)
            
            out = model.lm(
                input_ids=None,
                use_cache=True,
                attention_mask=attention_mask,
                output_hidden_states=True,
                inputs_embeds=last_text_embeds,
                past_key_values=past_key_values,
            )
            logits = out.logits
            past_key_values = out.past_key_values
        if logits is not None:
            last_token_logits = logits[0][-1]
            if temperature < 1e-4:
                token = int(torch.argmax(last_token_logits))
            else:
                probs = torch.softmax(last_token_logits / temperature, dim=-1)
                token = int(torch.multinomial(probs, num_samples=1))
            output_ids.append(token)
        ret = tokenizer.decode(token)
        if ret == '<box_begin>':
            attention_mask = torch.ones(1, past_key_values[0][0].shape[-2] + 1, device=device)
            bbox_embeds = model.box_token.bfloat16()
            out = model.lm(
                inputs_embeds=bbox_embeds,
                use_cache=True,
                attention_mask=attention_mask,
                output_hidden_states=True,
                past_key_values=past_key_values
            )
            last_hidden_states = out.hidden_states[-1]
            selected_hidden_states = last_hidden_states[0][0]
            bbox.append(model.loc_decoder(selected_hidden_states))
            last_token_logits = logits[0][-1]
            if temperature < 1e-4:
                token = int(torch.argmax(last_token_logits))
            else:
                probs = torch.softmax(last_token_logits / temperature, dim=-1)
                token = int(torch.multinomial(probs, num_samples=1))
        if ret == '<track_begin>':
            attention_mask = torch.ones(1, past_key_values[0][0].shape[-2] + 1, device=device)
            tracking_embeds = model.track_token
            out = model.lm(
                inputs_embeds=tracking_embeds,
                use_cache=True,
                attention_mask=attention_mask,
                output_hidden_states=True,
                past_key_values=past_key_values
            )
            last_hidden_states = out.hidden_states[-1]
            selected_hidden_states = last_hidden_states[0][0].to(dtype = torch.bfloat16)
            
            embed_sam_boxes = model.track_embed(selected_hidden_states).reshape(1, 3, 256)
            
            inference_state = model.sam.init_state_images(sam_images, size_hw[0][0], size_hw[0][1])
            gt_embeds = model.sam.get_prompt_embeding(inference_state, None, None, False, input_boxes[0].cuda(), device = text_embeds.device) 
            ann_frame_idx = 0
            ann_obj_id = 0
            box = np.array([0, 0, 0, 0], dtype=np.float32)
            _, out_obj_ids, out_mask_logits = model.sam.add_new_box_embeding(
                inference_state=inference_state,
                frame_idx=ann_frame_idx,
                obj_id=ann_obj_id,
                box=box,
                box_embeding=embed_sam_boxes,
            )
            video_segments = {}  # video_segments contains the per-frame segmentation results
            for out_frame_idx, out_obj_ids, out_mask_logits in model.sam.propagate_in_video(inference_state):
                video_segments[out_frame_idx] = {
                    out_obj_id: (out_mask_logits[i] > 0.0)
                    for i, out_obj_id in enumerate(out_obj_ids)
                }
            video_segments = [video_segments[tt][0] for tt in video_segments]
            # bbox = model.find_boundaries_torch(video_segments[0].squeeze(0).cpu())
            return ret, [], video_segments

        if (ret == '</s>'):
            break
    ret = tokenizer.decode(output_ids)
    del past_key_values
    return ret, bbox, video_segments
# dict_keys(['input_ids', 'attention_mask', 'labels', 'instruction', 'video', 'video_idx', 'sec', 'vlen', 'frame_indices', 'fps', 'input_boxes', 'gt_masks', 'sam_images', 'size_hw'])

import json
from tqdm import tqdm
for idx, dd in tqdm(enumerate(dataloader)):
    ret, bbox, video_segments = generate(dd['input_ids'].cuda(), dd['attention_mask'].cuda(), dd['labels'].cuda(),None, None, dd['video'].to(dtype=torch.bfloat16).cuda(), dd['video_idx'].cuda(), dd['input_boxes'], dd['size_hw'], dd['sam_images'].cuda().to(dtype=torch.bfloat16))
    output_file = "segment_test.jsonl"
    prediction = {"video": dd['path'][0], 'mask_path':dd['mask_path'][0], 
                    "pred": str(mask_util.encode(np.asfortranarray(np.array(torch.stack(video_segments, dim=-1).squeeze(0).cpu()))))
                    }
    with open(output_file, 'a') as f:
        json.dump(prediction, f)
        f.write('\n')


In [26]:
video.shape

torch.Size([1, 3, 16, 224, 224])